# Leveraging LLMs for Top-Down Sector Allocation in Automated Trading

This notebook implements the strategy described in the paper "Leveraging LLMs for Top-Down Sector Allocation in Automated Trading" by Ryan Quek Wei Heng, Edoardo Vittori, Keane Ong, Rui Mao, Erik Cambria, and Gianmarco Mengaldo, published on March 12, 2025 (arXiv: https://arxiv.org/abs/2503.09647).

The strategy involves using Large Language Models (LLMs) to analyze macroeconomic conditions and market sentiment for sector-level portfolio allocation. The framework processes multiple data streams, including policy documents, economic indicators, and sentiment patterns, to make informed sector allocation decisions.

## Abstract
The paper introduces a methodology leveraging Large Language Models (LLMs) for sector-level portfolio allocation through systematic analysis of macroeconomic conditions and market sentiment. The framework emphasizes top-down sector allocation by processing multiple data streams simultaneously, including policy documents, economic indicators, and sentiment patterns. Empirical results demonstrate superior risk-adjusted returns compared to traditional cross momentum strategies, achieving a Sharpe ratio of 2.51 and portfolio return of 8.79% versus -0.61 and -1.39% respectively. These results suggest that LLM-based systematic macro analysis presents a viable approach for enhancing automated portfolio allocation decisions at the sector level.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we define the configuration parameters, including the ticker universe, strategy parameters, and our hypothesis.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'FB', 'TSLA', 'NVDA', 'NFLX', 'BRK-B', 'JPM']
RISK_FREE_RATE = 0.02
REBALANCING_FREQUENCY = 'M'  # Monthly rebalancing

# Hypothesis
# We hypothesize that using LLMs to analyze macroeconomic conditions and market sentiment
# will provide superior sector allocation decisions leading to higher risk-adjusted returns.

## Phase 2 — Data Download & Feature Computation

In this phase, we download market data for the ticker universe and compute necessary features and factors.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download market data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-01-01', group_by='ticker')
prices = pd.DataFrame({ticker: data[ticker]['Adj Close'] for ticker in UNIVERSE})
returns = prices.pct_change().dropna()

# Feature computation (example: momentum factor)
momentum = returns.rolling(window=12).mean()

# Cross-sectional normalization
momentum_normalized = momentum.rank(axis=1, pct=True)

## Phase 3 — Signal Generation & Portfolio Construction

In this phase, we generate trading signals, size positions, and construct the portfolio.

In [ ]:
# Signal generation
signals = momentum_normalized.shift(1)

# Position sizing (equal-weighted long-short)
long_signals = signals > 0.5
short_signals = signals < 0.5
weights = pd.DataFrame(np.where(long_signals, 1/long_signals.sum(axis=1), 0), index=signals.index, columns=signals.columns)
weights -= pd.DataFrame(np.where(short_signals, 1/short_signals.sum(axis=1), 0), index=signals.index, columns=signals.columns)

## Phase 4 — Vectorized Backtest

In this phase, we perform a vectorized backtest of the strategy, ensuring no look-ahead bias.

In [ ]:
# Vectorized backtest
portfolio_returns = (returns * weights).sum(axis=1)
cumulative_returns = (1 + portfolio_returns).cumprod()

# Plot cumulative returns
import matplotlib.pyplot as plt

plt.plot(cumulative_returns)
plt.title('Cumulative Returns')
plt.xlabel('Date')
plt.ylabel('Returns')
plt.show()

## Phase 5 — Performance Metrics

In this phase, we calculate performance metrics such as Sharpe, Sortino, Calmar ratios, max drawdown, and plot the equity curve.

In [ ]:
from scipy.stats import norm

# Performance metrics
annual_return = portfolio_returns.mean() * 12
annual_volatility = portfolio_returns.std() * np.sqrt(12)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_volatility
sortino_ratio = (annual_return - RISK_FREE_RATE) / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(12)
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = annual_return / (-max_drawdown)

print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 6 — Monitoring Stub

In this phase, we create a function to print daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor_portfolio(prices, weights):
    current_prices = prices.iloc[-1]
    current_weights = weights.iloc[-1]
    daily_pnl = (current_prices / prices.iloc[-2] - 1) * current_weights
    total_pnl = daily_pnl.sum()
    print(f'Daily P&L: {total_pnl:.2%}')
    print('Current Positions:')
    print(current_weights[current_weights!= 0])

# Example usage
monitor_portfolio(prices, weights)